In [ ]:
# MCP SCENARIO: “Smart IT Helpdesk Assistant”
# 🧩 Scenario Background

# You are working in a company called ABC Corp.

# Employees face issues like:

# VPN not working
# Printer not responding
# Software errors

# 👉 Instead of calling IT support, employees use an AI Helpdesk Bot.

# 🤖 What this Bot Should Do

# When a user types a problem:

# Understand the issue
# Decide if a ticket is needed
# Identify:
# Category (Network / Hardware / General)
# Priority (High / Medium)
# Create a ticket
# Show confirmation
# 🧠 How MCP Fits Here
# Component	Role in Scenario
# Agent	Helpdesk Bot
# MCP Layer	Decision + Tool calling
# Tool	Ticket Creation System
# User	Employee




# ============================================
# STEP 0: DATABASE (Simulated storage)
# ============================================

tickets_db = []  # This stores all tickets


# ============================================
# STEP 1: TOOL (MCP TOOL)
# ============================================

def create_ticket(issue, priority, category):
    """
    This function simulates a TOOL in MCP
    In real world → API / Database / ServiceNow
    """

    ticket_id = f"INC{1000 + len(tickets_db)}"

    ticket = {
        "ticket_id": ticket_id,
        "issue": issue,
        "priority": priority,
        "category": category
    }

    tickets_db.append(ticket)

    return ticket


# ============================================
# STEP 2: AGENT REASONING (LLM SIMULATION)
# ============================================

def analyze_input(user_input):
    """
    Simulates how an LLM understands user input
    Extracts:
    - category
    - priority
    """

    text = user_input.lower()

    # 🔹 Category Detection
    if "vpn" in text:
        category = "network"
    elif "printer" in text:
        category = "hardware"
    elif "email" in text:
        category = "software"
    else:
        category = "general"

    # 🔹 Priority Detection
    if "urgent" in text or "immediately" in text:
        priority = "high"
    elif "slow" in text:
        priority = "low"
    else:
        priority = "medium"

    return category, priority


# ============================================
# STEP 3: DECISION ENGINE (MCP CORE)
# ============================================

def should_call_tool(user_input):
    """
    Decides whether to call a tool or not
    This is MCP decision layer
    """

    keywords = ["issue", "problem", "ticket", "not working"]

    return any(word in user_input.lower() for word in keywords)


# ============================================
# STEP 4: MCP ORCHESTRATOR
# ============================================

def mcp_agent(user_input):
    """
    This is the MAIN MCP FLOW
    It connects:
    Agent → Decision → Tool → Response
    """

    print("\n🧠 Agent received input:", user_input)

    # STEP 4.1: Decision
    if should_call_tool(user_input):

        print("➡️ Decision: Tool call required")

        # STEP 4.2: Analyze input
        category, priority = analyze_input(user_input)

        print(f"📊 Extracted → Category: {category}, Priority: {priority}")

        # STEP 4.3: Prepare payload (MCP format)
        payload = {
            "issue": user_input,
            "priority": priority,
            "category": category
        }

        print("📦 MCP Payload:", payload)

        # STEP 4.4: Call tool
        result = create_ticket(**payload)

        print("⚙️ Tool executed successfully")

        # STEP 4.5: Final response
        return f"""
        ✅ Ticket Created Successfully!

        Ticket ID: {result['ticket_id']}
        Issue: {result['issue']}
        Category: {result['category']}
        Priority: {result['priority']}
        """

    else:
        print("➡️ Decision: No tool needed (AI response)")

        return "🤖 AI Response: Please describe your issue clearly."


# ============================================
# STEP 5: RUN INTERACTIVE LOOP
# ============================================

print("🚀 MCP Demo Started (Type 'exit' to stop)\n")

while True:

    user_input = input("Enter your query: ")

    if user_input.lower() == "exit":
        print("👋 Exiting MCP demo...")
        break

    response = mcp_agent(user_input)
    print(response)


In [ ]:
# ============================================
# SETUP
# ============================================

import os
import json
from groq import Groq
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

api_key = os.getenv("GROQ_API_KEY")

if not api_key:
    raise ValueError("❌ GROQ_API_KEY not found in .env file")

client = Groq(api_key=api_key)

# ============================================
# DATABASE
# ============================================

tickets_db = []

# ============================================
# TOOL: CREATE TICKET
# ============================================

def create_ticket(issue, priority, category):
    ticket_id = f"INC{1000 + len(tickets_db)}"

    ticket = {
        "ticket_id": ticket_id,
        "issue": issue,
        "priority": priority,
        "category": category
    }

    tickets_db.append(ticket)
    return ticket


# ============================================
# LLM ANALYSIS
# ============================================

def analyze_with_llm(user_input):

    prompt = f"""
You are an IT helpdesk assistant.

Analyze the user issue and respond STRICTLY in JSON format:

{{
  "create_ticket": true/false,
  "category": "network/hardware/software/general",
  "priority": "high/medium/low"
}}

User Input: "{user_input}"
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    output = response.choices[0].message.content.strip()

    try:
        parsed = json.loads(output)
    except Exception as e:
        print("⚠️ JSON parsing failed, fallback triggered")
        parsed = {
            "create_ticket": True,
            "category": "general",
            "priority": "medium"
        }

    return parsed


# ============================================
# MCP AGENT
# ============================================

def mcp_agent(user_input):

    print("\n🧠 Agent received:", user_input)

    decision = analyze_with_llm(user_input)

    print("🤖 LLM Decision:", decision)

    if decision["create_ticket"]:

        payload = {
            "issue": user_input,
            "priority": decision["priority"],
            "category": decision["category"]
        }

        print("📦 MCP Payload:", payload)

        result = create_ticket(**payload)

        return f"""
✅ Ticket Created Successfully!

Ticket ID: {result['ticket_id']}
Issue: {result['issue']}
Category: {result['category']}
Priority: {result['priority']}
"""

    else:
        return "🤖 AI Response: No ticket required. Try basic troubleshooting."


# ============================================
# RUN LOOP (Jupyter + VS Code Friendly)
# ============================================

print("🚀 LLM MCP Helpdesk Started (type 'exit')\n")

while True:
    user_input = input("Enter issue: ")

    if user_input.lower() == "exit":
        print("👋 Exiting...")
        break

    response = mcp_agent(user_input)
    print(response)

🚀 LLM MCP Helpdesk Started (type 'exit')


🧠 Agent received: vpn issue
🤖 LLM Decision: {'create_ticket': True, 'category': 'network', 'priority': 'medium'}
📦 MCP Payload: {'issue': 'vpn issue', 'priority': 'medium', 'category': 'network'}

✅ Ticket Created Successfully!

Ticket ID: INC1000
Issue: vpn issue
Category: network
Priority: medium



In [ ]:
# MCP SCENARIO: “Smart HR Onboarding Assistant”
# 🧩 Scenario Background
# You are working in a company called XYZ Corp.
# New employees often face challenges during onboarding, such as:
# - Trouble accessing payroll portal
# - Confusion about leave policies
# - Difficulty setting up email accounts
# - Questions about training schedules
# 👉 Instead of emailing HR or waiting for responses, employees use an AI Onboarding Bot.

# 🤖 What this Bot Should Do
# When a new hire types a question/problem:
# - Understand the query (e.g., “I can’t log into payroll”)
# - Decide if escalation to HR is needed
# - Identify:
# - Category (Payroll / Policy / IT Setup / Training)
# - Priority (High / Medium)
# - Create a support ticket if required
# - Provide instant guidance (FAQs, step-by-step instructions)
# - Show confirmation and next steps

# 🧠 How MCP Fits Here
# |  |  | 
# |  |  | 
# |  |  | 
# |  |  | 
# |  |  | 



# This way, the MCP framework is reused in a Human Resources context, where the AI assistant streamlines onboarding, reduces HR workload, and ensures employees feel supported from day one.
# Would you like me to design another variation in a customer service setting (like retail or banking), so you can compare how MCP adapts across industries?

# ============================================
# RULE-BASED MCP HR ONBOARDING BOT
# ============================================

# DATABASE
tickets_db = []

# TOOL
def create_ticket(issue, category, priority):
    ticket_id = f"HR{1000 + len(tickets_db)}"

    ticket = {
        "ticket_id": ticket_id,
        "issue": issue,
        "category": category,
        "priority": priority
    }

    tickets_db.append(ticket)
    return ticket


# RULE ENGINE (NO LLM)
def analyze_issue(user_input):

    user_input = user_input.lower()

    if "payroll" in user_input or "salary" in user_input:
        return True, "Payroll", "High"

    elif "leave" in user_input or "policy" in user_input:
        return False, "Policy", "Medium"

    elif "email" in user_input or "login" in user_input:
        return True, "IT Setup", "High"

    elif "training" in user_input:
        return False, "Training", "Low"

    else:
        return True, "General", "Medium"


# AGENT
def hr_agent_rule_based(user_input):

    print("\n🧠 Received:", user_input)

    create, category, priority = analyze_issue(user_input)

    print("📊 Decision:", create, category, priority)

    if create:
        ticket = create_ticket(user_input, category, priority)

        return f"""
✅ Ticket Created!

ID: {ticket['ticket_id']}
Category: {category}
Priority: {priority}
"""

    else:
        return f"""
🤖 Guidance:
This seems informational ({category}).
Please check HR portal or FAQs.
"""


# RUN
print("🚀 HR Bot (Rule-Based)")

while True:
    user_input = input("Enter issue (exit to quit): ")

    if user_input.lower() == "exit":
        break

    print(hr_agent_rule_based(user_input))

🚀 HR Bot (Rule-Based)

🧠 Received: i cant log into payroll
📊 Decision: True Payroll High

✅ Ticket Created!

ID: HR1000
Category: Payroll
Priority: High



In [ ]:
# with llm
# ============================================
# LLM-BASED MCP HR ONBOARDING BOT
# ============================================

import os
import json
from groq import Groq
from dotenv import load_dotenv

# LOAD ENV
load_dotenv()
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

# DATABASE
tickets_db = []

# TOOL
def create_ticket(issue, category, priority):
    ticket_id = f"HR{1000 + len(tickets_db)}"

    ticket = {
        "ticket_id": ticket_id,
        "issue": issue,
        "category": category,
        "priority": priority
    }

    tickets_db.append(ticket)
    return ticket


# LLM ANALYSIS
def analyze_with_llm(user_input):

    prompt = f"""
You are an HR onboarding assistant.

Analyze the user query and respond in JSON:

{{
  "create_ticket": true/false,
  "category": "Payroll/Policy/IT Setup/Training/General",
  "priority": "High/Medium/Low"
}}

User Query: "{user_input}"
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    output = response.choices[0].message.content.strip()

    try:
        return json.loads(output)
    except:
        return {
            "create_ticket": True,
            "category": "General",
            "priority": "Medium"
        }


# AGENT
def hr_agent_llm(user_input):

    print("\n🧠 Received:", user_input)

    decision = analyze_with_llm(user_input)

    print("🤖 LLM Decision:", decision)

    if decision["create_ticket"]:

        ticket = create_ticket(
            user_input,
            decision["category"],
            decision["priority"]
        )

        return f"""
✅ Ticket Created!

ID: {ticket['ticket_id']}
Category: {ticket['category']}
Priority: {ticket['priority']}

📌 HR will contact you shortly.
"""

    else:
        return f"""
🤖 Instant Help:
This is a {decision['category']} query.
Please check onboarding docs or HR portal.
"""


# RUN
print("🚀 HR Bot (LLM-Based)")

while True:
    user_input = input("Enter issue (exit to quit): ")

    if user_input.lower() == "exit":
        break

    print(hr_agent_llm(user_input))

🚀 HR Bot (LLM-Based)

🧠 Received: i cant log into payroll
🤖 LLM Decision: {'create_ticket': True, 'category': 'General', 'priority': 'Medium'}

✅ Ticket Created!

ID: HR1000
Category: General
Priority: Medium

📌 HR will contact you shortly.



In [ ]:
# MCP SCENARIO: “Smart Banking Support Assistant”
# 🧩 Scenario Background
# You are working in a company called FinTrust Bank.
# Customers often face issues such as:
# - Credit card not working
# - Trouble with online banking login
# - Queries about loan status
# - Transaction disputes
# 👉 Instead of calling customer care, customers use an AI Banking Support Bot.

# 🤖 What this Bot Should Do
# When a customer types a problem:
# - Understand the issue (e.g., “My card was declined”)
# - Decide if escalation to a human agent is needed
# - Identify:
# - Category (Card Services / Online Banking / Loans / Transactions)
# - Priority (High / Medium)
# - Create a support ticket if required
# - Provide instant guidance (FAQs, troubleshooting steps, policy info)
# - Show confirmation and next steps

# 🧠 How MCP Fits Here
# |  |  | 
# |  |  | 
# |  |  | 
# |  |  | 
# |  |  | 



# This way, MCP is applied in a financial services context, where the AI assistant reduces call center load, provides quick resolutions, and ensures customers feel supported with secure, reliable guidance.
# Would you like me to craft one more in a healthcare setting (like hospital patient support), so you can see how MCP adapts to critical service environments?

# ============================================
# MCP-BASED SMART BANKING SUPPORT ASSISTANT
# ============================================

import os
import json
from groq import Groq
from dotenv import load_dotenv

# ============================================
# SETUP
# ============================================

load_dotenv()

api_key = os.getenv("GROQ_API_KEY")

if not api_key:
    raise ValueError("❌ GROQ_API_KEY not found in .env")

client = Groq(api_key=api_key)

# ============================================
# DATABASE (MEMORY)
# ============================================

tickets_db = []

# ============================================
# TOOL: CREATE SUPPORT TICKET
# ============================================

def create_ticket(issue, category, priority):
    ticket_id = f"BANK{1000 + len(tickets_db)}"

    ticket = {
        "ticket_id": ticket_id,
        "issue": issue,
        "category": category,
        "priority": priority
    }

    tickets_db.append(ticket)
    return ticket


# LLM ANALYSIS (DECISION ENGINE)


def analyze_with_llm(user_input):

    prompt = f"""
You are a banking support assistant for FinTrust Bank.

Analyze the user query and respond STRICTLY in JSON:

{{
  "create_ticket": true/false,
  "category": "Card Services/Online Banking/Loans/Transactions/General",
  "priority": "High/Medium/Low"
}}

Guidelines:
- Card not working, login issues → High priority
- Loan queries → Medium
- General info → Low
- Fraud or unauthorized transactions → High priority + ticket

User Query: "{user_input}"
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    output = response.choices[0].message.content.strip()

    try:
        parsed = json.loads(output)
    except Exception:
        print(" JSON parsing failed, using fallback")
        parsed = {
            "create_ticket": True,
            "category": "General",
            "priority": "Medium"
        }

    return parsed

# ============================================
# AGENT (CONTROLLER)
# ============================================

def banking_agent(user_input):

    print("\n Received:", user_input)

    # Step 1: LLM Decision
    decision = analyze_with_llm(user_input)

    print("🤖 LLM Decision:", decision)

    # Step 2: Agent Control Logic
    if decision["create_ticket"]:

        payload = {
            "issue": user_input,
            "category": decision["category"],
            "priority": decision["priority"]
        }

        print("📦 MCP Payload:", payload)

        # Step 3: Tool Execution
        ticket = create_ticket(**payload)

        return f"""
Support Ticket Created!

Ticket ID: {ticket['ticket_id']}
Category: {ticket['category']}
Priority: {ticket['priority']}

📌 Our banking support team will contact you shortly.
"""

    else:
        return f"""
🤖 Instant Guidance:

This is a {decision['category']} query.
Please check:
- Mobile banking app help section
- FAQ on FinTrust portal

If issue persists, you can request escalation.
"""


# RUN LOOP


print(" Smart Banking Support Assistant Started (type 'exit')\n")

while True:

    user_input = input("Enter your issue: ")

    if user_input.lower() == "exit":
        print("👋 Exiting... Stay safe!")
        break

    response = banking_agent(user_input)
    print(response)

🏦 Smart Banking Support Assistant Started (type 'exit')


🧠 Received: my debit card is not working
🤖 LLM Decision: {'create_ticket': True, 'category': 'Card Services', 'priority': 'High'}
📦 MCP Payload: {'issue': 'my debit card is not working', 'category': 'Card Services', 'priority': 'High'}

✅ Support Ticket Created!

Ticket ID: BANK1000
Category: Card Services
Priority: High

📌 Our banking support team will contact you shortly.



In [ ]:
# Create a  Weather Tool MCP Server that any AI agent can use with sample use case